# Imports & paths

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from shapely import make_valid
from shapely.geometry import LineString, Polygon
from shapely.validation import explain_validity

os.environ.setdefault(
    "LOKY_MAX_CPU_COUNT",
    os.environ.get("NUMBER_OF_PROCESSORS", "1"),
)


def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "cnt_project").exists():
            return p

    raise RuntimeError(
        "Could not locate project root containing src/cnt_project"
    )


PROJECT_ROOT = _find_project_root(Path.cwd())

PROJECT_SRC_ROOT = PROJECT_ROOT / "src"
NOTEBOOK_ROOT = PROJECT_ROOT / "notebooks" / "paper_figures"

if str(PROJECT_SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_SRC_ROOT))

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))


from src.utils.paths import GT_TEST_ANNOTATIONS_PATH

from cnt_project.coco.overlay import overlay_annotations_on_ax

from cnt_project.features.core.cnt_feature_extraction import (
    _decode_annotation_mask,
    compute_line_densities_from_polygons,
    compute_line_densities_from_polygons_legacy_rasterized,
)

: 

# Define inputs

# 2. GT images takes line density from all rows and high density take line density of 3 rows

# stll uses the legacy inputs

In [ ]:
ALL_ROWS = tuple(range(256))

test_structured_all_rows_legacy = build_test_set_structured(
    gt_json=GT_TEST_ANNOTATIONS_PATH_LEGACY,
    line_density_csv=LINE_DENSITY_CSV,
    rows=ALL_ROWS,
)

In [ ]:
display(
    test_structured_all_rows_legacy[
        [
            "filename",
            "mean_line_density_image",
            "manual_mean_line_density_image",
            "line_density_per_micrometer",
            "manual_line_density_per_micrometer",
        ]
    ].head(10)
)

In [ ]:
df_test_all_rows = test_structured_all_rows_legacy.copy()
df_test_all_rows["dataset_type"] = "test"

In [ ]:
df_all_mixed = combine_test_and_high_density(
    test_set_structured=df_test_all_rows,
    high_density_df=df_high_3lines,
)

df_all_mixed = prepare_operational_limit_dataframe(
    df_all_mixed
)

In [ ]:
df_all_mixed["reference_method"] = (
    df_all_mixed["dataset_type"]
    .map(
        {
            "test": "GT, 256 rows",
            "high_density": "manual, 3 rows",
        }
    )
)

In [ ]:
print(
    df_all_mixed["reference_method"]
    .value_counts()
)

In [ ]:
sampling_comparison = (
    test_structured_3lines[
        [
            "filename",
            "manual_line_density_per_micrometer",
        ]
    ]
    .rename(
        columns={
            "manual_line_density_per_micrometer":
                "gt_density_3lines_um"
        }
    )
    .merge(
        test_structured_all_rows_legacy[
            [
                "filename",
                "manual_line_density_per_micrometer",
            ]
        ].rename(
            columns={
                "manual_line_density_per_micrometer":
                    "gt_density_256rows_um"
            }
        ),
        on="filename",
        how="inner",
    )
)
sampling_comparison["signed_difference_um"] = (
    sampling_comparison["gt_density_3lines_um"]
    - sampling_comparison["gt_density_256rows_um"]
)

sampling_comparison["absolute_difference_um"] = (
    sampling_comparison["signed_difference_um"].abs()
)

sampling_comparison["relative_difference_pct"] = (
    sampling_comparison["absolute_difference_um"]
    / sampling_comparison["gt_density_256rows_um"]
    * 100.0
)


In [ ]:
display(sampling_comparison)

In [ ]:
print(
    sampling_comparison[
        [
            "absolute_difference_um",
            "relative_difference_pct",
        ]
    ].describe()
)
sampling_comparison[
    [
        "filename",
        "gt_density_3lines_um",
        "gt_density_256rows_um",
        "absolute_difference_um",
        "relative_difference_pct",
    ]
]

In [ ]:
fullrow_prediction_error = test_structured_all_rows_legacy[
    [
        "filename",
        "line_density_per_micrometer",
        "manual_line_density_per_micrometer",
    ]
].copy()

fullrow_prediction_error = fullrow_prediction_error.rename(
    columns={
        "line_density_per_micrometer": "predicted_density_256rows_um",
        "manual_line_density_per_micrometer": "gt_density_256rows_um",
    }
)

fullrow_prediction_error["signed_error_um"] = (
    fullrow_prediction_error["predicted_density_256rows_um"]
    - fullrow_prediction_error["gt_density_256rows_um"]
)

fullrow_prediction_error["absolute_error_um"] = (
    fullrow_prediction_error["signed_error_um"].abs()
)

fullrow_prediction_error["relative_error_pct"] = (
    fullrow_prediction_error["absolute_error_um"]
    / fullrow_prediction_error["gt_density_256rows_um"]
    * 100.0
)

fullrow_prediction_error = fullrow_prediction_error.sort_values(
    "relative_error_pct",
    ascending=False,
)

display(fullrow_prediction_error)

In [ ]:
SAVE_MIXED_LINEAR = (
    OUTPUTS["density"]
    / "operational_limit_mixed_reference"
)

SAVE_MIXED_LOG = (
    OUTPUTS["density"]
    / "operational_limit_log_mixed_reference"
)

SAVE_MIXED_RELATIVE = (
    OUTPUTS["density"]
    / "operational_limit_relative_density_error_mixed_reference"
)

SAVE_MIXED_BA = (
    OUTPUTS["density"]
    / "operational_limit_bland_altman_density_mixed_reference"
)

In [ ]:
out_svg_mixed_relative = plot_relative_density_error(
    df_all_mixed,
    save_path=SAVE_MIXED_RELATIVE,
)

print("Saved:", out_svg_mixed_relative)

In [ ]:
out_svg_mixed_ba = plot_bland_altman_density(
    df_all_mixed,
    save_path=SAVE_MIXED_BA,
)

print("Saved:", out_svg_mixed_ba)

In [ ]:
out_svg_mixed_linear = plot_operational_limit(
    df_all_mixed,
    save_path=SAVE_MIXED_LINEAR,
)

print("Saved:", out_svg_mixed_linear)

In [ ]:
out_svg_mixed_log = plot_operational_limit_log_scale(
    df_all_mixed,
    save_path=SAVE_MIXED_LOG,
)

print("Saved:", out_svg_mixed_log)